In [27]:
import xarray as xr
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import TransformedTargetRegressor
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.inspection import permutation_importance


In [28]:
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.375, -60.0, -44.875]
min_time, max_time = pd.to_datetime("2013-01-01"), pd.to_datetime("2023-12-31")

fishing_ds = xr.open_dataset("../data/processed/targets/cpue_HKP.nc")

fishing = fishing_ds["CPUE"]
fishing = fishing.fillna(0)


temp_ds = xr.open_dataset("../data/processed/dynamic/to_surface.nc")
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_lag1 = temp.shift(time=1).fillna(0)

temp_bottom_ds = xr.open_dataset("../data/processed/dynamic/temp_bottom.nc")
temp_bottom = temp_bottom_ds["to"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

temp_bottom_lag1 = temp_bottom.shift(time=1).fillna(0)

chl_ds = xr.open_dataset("../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

chl_lag1 = chl.shift(time=1).fillna(0)

mixed_ds = xr.open_dataset("../data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

mixed_lag1 = mixed.shift(time=1).fillna(0)

depth_ds = xr.open_dataset("../data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../data/processed/dynamic/zo_surface.nc")
zo = zo_ds["zo"]
zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

zo_lag1 = zo.shift(time=1).fillna(0)

so_ds = xr.open_dataset("../data/processed/dynamic/so_surface.nc")
so = so_ds["so"]
so = (so - so.mean()) / so.std()
so = so.fillna(0)

mask_ds = xr.open_dataset("../data/processed/static/fishing_area_mask.nc")
mask = mask_ds["mask"]
mask = mask.broadcast_like(temp)

month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)
month = month.broadcast_like(temp)

year = temp["time"].dt.year
year = (year - year.mean()) / year.std()
year = year.broadcast_like(temp)

lat = (temp["lat"] - temp["lat"].mean()) / temp["lat"].std()
lon = (temp["lon"] - temp["lon"].mean()) / temp["lon"].std()
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp) #se añade como dinámica porque ya se ha corregido la forma

temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1, zo_lag1 = xr.align(temp, temp_bottom, chl, mixed, fishing, mask, depth, month_sin, month_cos, lat, lon, zo, so, month, year, temp_lag1, mixed_lag1, temp_bottom_lag1, chl_lag1, zo_lag1, join="inner")

cropped = lambda da: da.sel(
    lon=slice(min_lon, max_lon),
    lat=slice(min_lat, max_lat),
    time=slice(min_time, max_time)
)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
mixed_lag1 = cropped(mixed_lag1)
fishing = cropped(fishing)
mask = cropped(mask)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)
month = cropped(month)
month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)
year = cropped(year)
temp_lag1 = cropped(temp_lag1)
temp_bottom_lag1 = cropped(temp_bottom_lag1)
chl_lag1 = cropped(chl_lag1)
zo_lag1 = cropped(zo_lag1)

In [29]:
y = fishing  # (time, lat, lon)
y = y.transpose("time", "lat", "lon")

vars_ = [temp, so, mixed, zo, temp_bottom,  
        chl,
        month_sin, month_cos,
        lat, lon,
        depth, year,
        ]
vars_names = [v.name for v in vars_]
in_channels = len(vars_)
X = xr.concat(vars_, dim="channel")
X = X.transpose("time", "channel", "lat", "lon")

split_year = 2020

train_X = X.sel(time=slice(None, f"{split_year-1}-12-31"))
test_X  = X.sel(time=slice(f"{split_year}-01-01", None))

train_y = y.sel(time=slice(None, f"{split_year-1}-12-31"))
test_y  = y.sel(time=slice(f"{split_year}-01-01", None))


def create_windows(X, y, window=10):
    X_data = X.values   # (time, channels, H, W)
    y_data = y.values   # (time, H, W)

    X_seq, y_seq= [], []

    for i in range(len(X_data) - window):
        X_seq.append(X_data[i:i+window]) #12 months
        y_seq.append(y_data[i+window]) #next month
        

    return (
        torch.tensor(np.stack(X_seq), dtype=torch.float32),
        torch.tensor(np.stack(y_seq), dtype=torch.float32),
    )

window = 12 #12 months

X_train, y_train = create_windows(train_X, train_y, window)
X_test, y_test= create_windows(test_X, test_y, window)

print(X_train.shape)  # (N, T, C, H, W)
print(y_train.shape)  # (N, H, W)
###### Data Loaders ######
batch_size = 24

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=batch_size,
    shuffle=False
)


torch.Size([72, 12, 12, 21, 8])
torch.Size([72, 21, 8])


In [30]:
def dl_to_rf(X, y):
    # X: (N, T, C, H, W)
    N, T, C, H, W = X.shape

    # Move spatial dims forward: (N, H, W, T, C)
    X = X.permute(0, 3, 4, 1, 2)

    # Flatten per pixel
    X_rf = X.reshape(N * H * W, T * C)

    # Target per pixel
    y_rf = y.reshape(N * H * W)

    return X_rf, y_rf

X_rf_train, y_rf_train = dl_to_rf(X_train, y_train)
X_rf_test, y_rf_test = dl_to_rf(X_test, y_test)

print(X_rf_train.shape)
print(y_rf_train.shape)

torch.Size([12096, 144])
torch.Size([12096])


In [ ]:


rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_rf_train, y_rf_train)

pred = rf.predict(X_rf_test)

rmse = np.sqrt(mean_squared_error(y_rf_test, pred))
mae = mean_absolute_error(y_rf_test, pred)
r2 = r2_score(y_rf_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)


# ---------------------------
# FEATURE IMPORTANCE FIX
# ---------------------------

importances = rf.feature_importances_
T = window
C = len(vars_names)
# reshape flat importance → (time, variables)
imp_matrix = importances.reshape(T, C)
# aggregate over time → (variables,)
var_importance = imp_matrix.sum(axis=0)
# build clean table
feature_importance_df = pd.DataFrame({
    "feature": vars_names,
    "importance": var_importance
}).sort_values("importance", ascending=False)
print(feature_importance_df)

RMSE: 1.7447524406097867
MAE: 1.354149382717977
R2: 0.452671079433025
   feature  importance
10   depth    0.150150
3       zo    0.143062
9      lon    0.130201
8      lat    0.128969
0       to    0.117436
2   mlotst    0.083720
1       so    0.077961
5      CHL    0.076333
4       to    0.074571
11    year    0.008244
6    month    0.005213
7    month    0.004141


In [33]:
gbr = HistGradientBoostingRegressor(
    max_depth=20,
    learning_rate=0.1,
    max_iter=100,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42,
)

gbr.fit(X_rf_train, y_rf_train)

pred = gbr.predict(X_rf_test)

rmse = np.sqrt(mean_squared_error(y_rf_test, pred))
mae = mean_absolute_error(y_rf_test, pred)
r2 = r2_score(y_rf_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)


RMSE: 1.5789439813925408
MAE: 1.0817247020672252
R2: 0.5517562818273523
